# Episode 8: Assembly & Policies

Every turn, something has to decide *what the model actually sees*: which past messages, how much history, what extra context. That decision is **assembly**, and you control it with **policies**.

**In this episode you'll see:** an assembly policy change what an agent can remember.

## What assembly does

Before each LLM call, the agent builds the request from two things:

- **prompts** — the system prompt and any injected context
- **events** — the conversation history so far

An **assembly policy** transforms that `(prompts, events)` pair. You pass policies to the `assembly=` slot, and they run in order, each piping into the next.

## Two kinds of policy

| Kind | Does | Built-ins |
|---|---|---|
| **Injection** | Adds to `prompts` | `WorkingMemoryPolicy`, `EpisodicMemoryPolicy`, `AlertPolicy` |
| **Reduction** | Trims `events` | `ConversationPolicy`, `SlidingWindowPolicy`, `TokenBudgetPolicy` |

Order matters: **inject before you reduce**, so reduction sees the final picture.

This episode demonstrates a reduction policy — `SlidingWindowPolicy` — because its effect is easy to *see*.

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.policies import SlidingWindowPolicy

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)


## Demo: a sliding window makes the agent forget

`SlidingWindowPolicy(max_events=4)` keeps only the **last 4 events** in view each turn. We tell the agent a secret code, chat for a few turns, then ask it to recall the code — by then, the turn where we said it has slid out of the window.

`transparent=True` adds a small note to the prompt about how many events were kept — useful while tuning.

In [1]:
windowed = Agent(
    "windowed",
    prompt="You are a helpful assistant. Answer in one short sentence.",
    config=config,
    assembly=[SlidingWindowPolicy(max_events=4, transparent=True)],
)

reply = await windowed.ask("Remember this: the secret code is BANANA42.")
print("turn 1:", reply.body)
reply = await reply.ask("What is the capital of France?")
print("turn 2:", reply.body)
reply = await reply.ask("What is 10 times 10?")
print("turn 3:", reply.body)
reply = await reply.ask("What is the secret code I told you?")
print("turn 4:", reply.body)


turn 1: Got it, the secret code is BANANA42.
turn 2: The capital of France is Paris.
turn 3: 10 times 10 is 100.
turn 4: I’m sorry, but I don’t have any record of a secret code you mentioned.


## The comparison: full history remembers

Same conversation, same questions — but no assembly policy, so the agent sees the *entire* history. It recalls the code with no trouble.

In [2]:
full = Agent(
    "full",
    prompt="You are a helpful assistant. Answer in one short sentence.",
    config=config,
)

reply = await full.ask("Remember this: the secret code is BANANA42.")
reply = await reply.ask("What is the capital of France?")
reply = await reply.ask("What is 10 times 10?")
reply = await reply.ask("What is the secret code I told you?")
print("turn 4:", reply.body)


turn 4: The secret code you told me is BANANA42.


## The other policies

`SlidingWindowPolicy` is one of six built-ins in `autogen.beta.policies`:

- **`ConversationPolicy()`** — drops non-conversation events (lifecycle, observer) from the model's view
- **`TokenBudgetPolicy(max_tokens=N)`** — trims by approximate token count instead of event count
- **`WorkingMemoryPolicy()`** — injects persistent memory from a knowledge store (Episode 19)
- **`EpisodicMemoryPolicy()`** — injects summaries of past conversations (Episode 19)
- **`AlertPolicy()`** — delivers observer alerts to the model (Episode 25)

You stack them: `assembly=[WorkingMemoryPolicy(), AlertPolicy(), SlidingWindowPolicy(max_events=80)]`.

## Additional content

**Injection before reduction.** If a reduction policy runs before an injection policy, the injected context is never counted against the budget — usually not what you want. Keep injection policies first in the list.

**`transparent=True`** appends a short `[policy] Showing X of Y events.` note to the prompt. Turn it on while tuning so you can see what each policy did; turn it off in production.

**Middleware vs assembly.** Episode 7's `HistoryLimiter` also trims history — but assembly policies are richer (token budgets, memory injection, alerts) and run as a dedicated stage. Reach for assembly when you're shaping *what the model sees*; reach for middleware for *behavior around the call*.

## What's next

You've shaped what goes *into* the agent. Next, Episode 9 shapes what comes *out* — **structured output**, where the agent returns a typed object instead of text.